In [ ]:
%load_ext noworkflow
nip = %now_ip

In [ ]:
def first(x):
    return next(x)

In [ ]:
def extract(trial_id, name, param, attr):
    t = nip.Trial(trial_id)
    
    # SQL query
    sql = first(nip.persistence.query("""
        SELECT f.id
        FROM function_activation f
        INNER JOIN object_value o
        ON f.id = o.function_activation_id
        WHERE f.name = '{}'
        AND o.value = '{}'
        AND f.trial_id = '{}'
    """.format(name, param, trial_id)))
    act_id = sql['id']
    
    # Prolog query
    prolog = first(t.trial_prolog.query("""
       access_id({0}, {1}, Y),
       hash_id({0}, Y, X, after)
    """.format(trial_id, act_id)))
    hash = prolog['X']
    
    # Create new file with old content
    content = nip.persistence.get(hash)
    with open('.temp.dat', 'w') as f:
        f.write(content)
        
    # Call external program to get result
    result = !./extractor .temp.dat $attr

    # Return result
    return '\n'.join(result)

In [ ]:
extract(1, 'z', '2', '1')

In [ ]:
extract(1, 'y', '1', 'ALL')

In [ ]:
extract(2, 'y', '1', 'ALL')